<a href="https://colab.research.google.com/github/Dkhan213/Etch-AI-Optimization/blob/main/AI_Optimization_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.1 MB/s eta 0:00:00


In [2]:
# ==========================================
# ENVIRONMENT SETUP
# ==========================================
# Quietly install the scikit-optimize machine learning library into Google Colab (-q means quiet)
!pip install scikit-optimize -q

import pandas as pd             # Pandas handles data tables (like reading CSV files)
import numpy as np              # Numpy handles complex math and array calculations
from skopt import Optimizer     # The core AI engine that performs Bayesian Optimization
from skopt.space import Space, Real  # Allows us to define physical variable ranges (continuous numbers)
import warnings

# Hide harmless background warnings from the libraries to keep the output clean
warnings.filterwarnings('ignore')


# ==========================================
# GITHUB DATA SOURCES
# ==========================================
# Direct links to raw GitHub CSV files so Colab can read them from anywhere
HISTORICAL_SOURCE = "https://raw.githubusercontent.com/dkhan213/etch-ai-optimization/main/historical_baseline.csv"
NANOFAB_SOURCE = "https://raw.githubusercontent.com/dkhan213/etch-ai-optimization/main/nanofab_experiments.csv"


# ==========================================
# STEP 1: DEFINING THE AI'S LEARNING SPACE
# ==========================================
# We define a deliberately wide search space here so the AI can mathematically digest
# the aggressive parameters found in the historical data without throwing an error.
# (We apply a strict safety filter for the real machine later in Step 3).
search_space = [
    Real(20.0, 160.0, name='rf_power'),    # Allows the AI to read paper data up to 160W
    Real(20.0, 50.0, name='sf6_flow'),     # Allows the AI to read paper data up to 50sccm
    Real(5.0, 50.0, name='o2_flow')        # Allows the AI to read paper data up to 50sccm
]


# ==========================================
# STEP 2: DATA INGESTION
# ==========================================
print("SYSTEM CHECK: Loading live data from GitHub...")

# Attempt to load the historical academic data
try:
    external_df = pd.read_csv(HISTORICAL_SOURCE)
except Exception as e:
    # If GitHub is down or the link breaks, create an empty table so the script doesn't crash
    print(f"Notice: Could not load historical baseline ({e}). Using empty DataFrame.")
    external_df = pd.DataFrame(columns=['rf_power', 'sf6_flow', 'o2_flow', 'etch_rate'])

# Attempt to load your actual cleanroom experiments
try:
    nanofab_df = pd.read_csv(NANOFAB_SOURCE)
except Exception:
    nanofab_df = pd.DataFrame(columns=['rf_power', 'sf6_flow', 'o2_flow', 'etch_rate'])

# ACTIVE LEARNING THRESHOLD LOGIC:
# This is the "transfer learning" switch. Once 3 actual runs in the cleanroom are logged,
# the AI drops the textbook data and trains exclusively on the machine's real-world behavior.
if len(nanofab_df) >= 3:
    print(f"Nanofab threshold reached ({len(nanofab_df)} runs logged). Training strictly on local hardware.")
    active_data = nanofab_df
else:
    print(f"Insufficient Nanofab data ({len(nanofab_df)} runs logged). Seeding prior with {len(external_df)} records.")
    active_data = external_df

# Isolate the 3 input variables (Features/X) as a list of lists.
# Pandas automatically ignores any extra columns like 'source'.
X_prior = active_data[['rf_power', 'sf6_flow', 'o2_flow']].values.tolist()

# Isolate the target objective (Etch Rate/Y).
# CRITICAL: skopt is designed to strictly MINIMIZE values.
# Because we want to MAXIMIZE etch rate, we multiply it by -1.
# (e.g., maximizing 1500 becomes minimizing -1500).
y_prior = (-active_data['etch_rate']).values.tolist()


# ==========================================
# STEP 3: THE ASK-AND-TELL OPTIMIZER
# ==========================================
print("\nCALCULATING: Generating optimal parameters...")

# Configure the Bayesian AI Engine
ai_engine = Optimizer(
    dimensions=search_space,  # Tells the AI the variable axes it is allowed to think about
    base_estimator="RF",      # Uses a "Random Forest" model to predict the etch rate landscape
    n_initial_points=0,       # Set to 0 to stop the AI from making random guesses; we want it to use our data immediately
    acq_func="EI",            # Expected Improvement: The math that balances exploring unknown areas vs exploiting known peaks
    random_state=42           # Locks the random seed so the script gives the exact same result if run twice
)

# Define the absolute physical safety limits for the Oxford System 100 to prevent photoresist burning
safe_limits = Space([
    Real(25.0, 60.0),  # Safe RF Power
    Real(20.0, 40.0),  # Safe SF6 Flow
    Real(5.0, 15.0)    # Safe O2 Flow
])

# Randomly generate 1,000 potential recipes that strictly obey the safety limits above
safe_candidates = safe_limits.rvs(1000, random_state=42)

# TELL: Feed the isolated data (X and Y) into the AI so it learns the physics
if len(X_prior) > 0:
    ai_engine.tell(X_prior, y_prior)

    # SAFETY OVERRIDE:
    # Instead of asking the AI to output an unbounded guess, we force it to look at our
    # 1,000 safe recipes and predict the etch rate for every single one.
    predicted_etch_rates = ai_engine.models[-1].predict(safe_candidates)

    # Find the specific recipe index that yielded the best predicted etch rate
    # (Using argmin because our etch rates are currently negative numbers)
    best_safe_idx = np.argmin(predicted_etch_rates)

    # Lock in that specific safe recipe as our next experiment
    next_experiment = safe_candidates[best_safe_idx]
else:
    # Fallback to the first safe candidate if both GitHub links fail entirely
    next_experiment = safe_candidates[0]


# ==========================================
# STEP 4: OUTPUT PROTOCOL
# ==========================================
print("\n--- TEAM ETCH-A-SKETCH: ML RECIPE FOR NEXT LAB ITERATION ---")
print("1. Set static parameters on the Oxford System 100:")
print("   -> ICP Source Power: 800.0 [W]")
print("   -> Chamber Pressure: 20.0  [mTorr]")
print("   -> Argon (Ar) Flow:  15.0  [SCCM]")
print("   -> Process Time:     120   [Seconds]")
print("\n2. Input the AI-OPTIMIZED variables:")
print(f"   -> RF Substrate Bias: {next_experiment[0]:.1f} [W]")
print(f"   -> SF6 Gas Flow:      {next_experiment[1]:.1f} [SCCM]")
print(f"   -> O2 Gas Flow:       {next_experiment[2]:.1f} [SCCM]")

SYSTEM CHECK: Loading live data from GitHub...
Insufficient Nanofab data (0 runs logged). Seeding prior with 8 records.

CALCULATING: Generating optimal parameters...

--- TEAM ETCH-A-SKETCH: ML RECIPE FOR NEXT LAB ITERATION ---
1. Set static parameters on the Oxford System 100:
   -> ICP Source Power: 800.0 [W]
   -> Chamber Pressure: 20.0  [mTorr]
   -> Argon (Ar) Flow:  15.0  [SCCM]
   -> Process Time:     120   [Seconds]

2. Input the AI-OPTIMIZED variables:
   -> RF Substrate Bias: 58.3 [W]
   -> SF6 Gas Flow:      30.8 [SCCM]
   -> O2 Gas Flow:       7.5 [SCCM]
